In [1]:
print("ok")

ok


In [2]:
import os
os.chdir('../')

In [3]:
"""
compute_fisher_ewc.py
─────────────────────────────────────────────────────────────────────────────
Diagonal Fisher Information Matrix computation for EWC (Elastic Weight
Consolidation) using a Whisper ASR model.

Fixes vs. original:
  1. Removed incorrect attention_mask (wrong shape; Whisper ignores it anyway).
  2. Loss is scaled by batch size before backward() so gradients are NOT
     divided across samples — avoids cross-sample cancellation.
  3. Fisher is normalised by total number of samples (not num_batches).
  4. tqdm progress bar added.
  5. Saves both fisher and theta_star in a single .pt file.
─────────────────────────────────────────────────────────────────────────────
"""

import torch
from torch.utils.data import DataLoader
from dataclasses import dataclass
from typing import Any

from datasets import load_dataset, Audio
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
)

try:
    from tqdm import tqdm
    USE_TQDM = True
except ImportError:
    USE_TQDM = False


# CONFIG

MODEL_NAME          = "openai/whisper-small"
CSV_PATH            = "Data/eval.csv"
AUDIO_DIR           = "Data/eval"
AUDIO_EXT           = ".flac"
MAX_SAMPLES         = 1000        # set to None to use full dataset
BATCH_SIZE          = 4
SAMPLING_RATE       = 16_000
OUTPUT_PATH         = "Data/EWC/ewc_fisher_whisper.pt"


# DEVICE

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# LOAD MODEL + PROCESSORS

print(f"Loading model: {MODEL_NAME}")

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
tokenizer         = WhisperTokenizer.from_pretrained(MODEL_NAME)
processor         = WhisperProcessor.from_pretrained(MODEL_NAME)

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)

# Eval mode: disables dropout so Fisher is computed at the MAP estimate.
# Gradients are still computed normally.
model.eval()


# LOAD DATASET
print("Loading dataset...")

dataset = load_dataset("csv", data_files=CSV_PATH)["train"]

# Build audio path column and cast to Audio so HF handles decoding + resampling
dataset = dataset.map(
    lambda x: {"audio_path": f"{AUDIO_DIR}/{x['utt_id']}{AUDIO_EXT}"}
)
dataset = dataset.cast_column("audio_path", Audio(sampling_rate=SAMPLING_RATE))

if MAX_SAMPLES is not None:
    dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))

print(f"Dataset size: {len(dataset)} samples")


# PREPROCESSING
def preprocess(batch):
    audio = batch["audio_path"]["array"]

    batch["input_features"] = feature_extractor(
        audio,
        sampling_rate=SAMPLING_RATE,
    ).input_features[0]                          # shape: (80, 3000)

    batch["labels"] = tokenizer(batch["transcript"]).input_ids

    return batch


dataset = dataset.map(
    preprocess,
    remove_columns=dataset.column_names,
    desc="Preprocessing",
)


# DATA COLLATOR

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    feature_extractor: Any
    tokenizer: Any
    decoder_start_token_id: int

    def __call__(self, features):
        # Encoder inputs
        input_features = [{"input_features": f["input_features"]} for f in features]

        batch = self.feature_extractor.pad(
            input_features,
            return_tensors="pt",
        )

        # WhisperForConditionalGeneration does not use it for fixed-length mel
        # spectrograms (shape [B, 80, 3000]), and adding a mask with the wrong
        # shape ([B, 80] instead of [B, 3000]) would cause silent errors.

        # Decoder labels 
        label_features = [{"input_ids": f["labels"]} for f in features]

        labels_batch = self.tokenizer.pad(
            label_features,
            return_tensors="pt",
        )

        # Replace padding token id with -100 so it is ignored by the loss
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        # Strip leading decoder_start_token if present (Whisper prepends it
        # automatically inside forward(); including it in labels double-counts)
        if (labels[:, 0] == self.decoder_start_token_id).all():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
    decoder_start_token_id=model.config.decoder_start_token_id,
)


# DATALOADER

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,          # order does not matter for Fisher; keep False for
                            # reproducibility
    collate_fn=data_collator,
    num_workers=0,          # set >0 if your environment supports multiprocessing
    pin_memory=(device.type == "cuda"),
)


# INITIALISE FISHER ACCUMULATORS

fisher: dict[str, torch.Tensor] = {
    name: torch.zeros_like(param, device=device)
    for name, param in model.named_parameters()
}


# FISHER COMPUTATION LOOP

# Theory recap:
#   Diagonal Fisher ≈ E[ (∂ log p(y|x,θ) / ∂θ)² ]
#   We approximate the expectation with a sample average over the dataset.
#
# Key correction:
#   HuggingFace returns loss = mean(NLL) over the batch.
#   To recover per-sample gradients (needed for an unbiased Fisher estimate),
#   we multiply loss by batch_size before calling backward().
#   This is equivalent to using reduction='sum' and then accumulating.

print("Computing Fisher Information Matrix...")

num_samples = 0
iterator = tqdm(loader, desc="Fisher") if USE_TQDM else loader

for step, batch in enumerate(iterator):
    batch = {
        k: v.to(device) if torch.is_tensor(v) else v
        for k, v in batch.items()
    }

    bs = batch["input_features"].shape[0]   # actual batch size (last batch may differ)
    num_samples += bs

    model.zero_grad()

    outputs = model(
        input_features=batch["input_features"],
        labels=batch["labels"],
    )

    # outputs.loss is mean NLL over non-ignored tokens in the batch.
    # Multiply by bs to approximate the sum so gradients are not divided
    # across samples — prevents cross-sample cancellation in Fisher estimate.
    loss = outputs.loss * bs
    loss.backward()

    # Accumulate squared gradients (diagonal Fisher approximation)
    for name, param in model.named_parameters():
        if param.grad is not None:
            fisher[name] += param.grad.detach() ** 2

    if not USE_TQDM and step % 50 == 0:
        print(f"  Batch {step:4d} / {len(loader)}  |  samples so far: {num_samples}")


# NORMALISE FISHER

# Divide by total number of samples (not num_batches) to obtain a proper
# per-sample expectation.
print(f"\nNormalising Fisher over {num_samples} samples...")

for name in fisher:
    fisher[name] /= num_samples


# SAVE θ* AND FISHER

# θ* = current model weights (the "anchor" point for EWC regularisation)
theta_star = {
    name: param.detach().cpu().clone()
    for name, param in model.named_parameters()
}

ewc_data = {
    "fisher":     {k: v.cpu() for k, v in fisher.items()},
    "theta_star": theta_star,
}

torch.save(ewc_data, OUTPUT_PATH)
print(f"Saved EWC data → {OUTPUT_PATH}")


# QUICK SANITY CHECK

total_params   = sum(p.numel() for p in model.parameters())
nonzero_fisher = sum(
    (v > 0).sum().item() for v in ewc_data["fisher"].values()
)
print(f"\nSanity check:")
print(f"  Total parameters        : {total_params:,}")
print(f"  Non-zero Fisher entries : {nonzero_fisher:,}")
print(f"  Coverage                : {100 * nonzero_fisher / total_params:.1f}%")
print("\nDone.")

/home/bishwa/Unversity/Audio_LLM_Memory/audio_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Loading model: openai/whisper-small
Loading dataset...


Map: 100%|██████████| 2620/2620 [00:00<00:00, 44457.61 examples/s]


Dataset size: 1000 samples


Preprocessing: 100%|██████████| 1000/1000 [00:18<00:00, 53.79 examples/s]


Computing Fisher Information Matrix...


Fisher: 100%|██████████| 250/250 [01:08<00:00,  3.67it/s]



Normalising Fisher over 1000 samples...
Saved EWC data → Data/EWC/ewc_fisher_whisper.pt

Sanity check:
  Total parameters        : 241,734,912
  Non-zero Fisher entries : 240,377,088
  Coverage                : 99.4%

Done.
